# Universal Multi-Stock Training (Full Universe)

Trains a single model on the full screened universe (491 symbols) using rigorous
per-symbol temporal splits from `data/splits.yml`. Evaluates on two separate held-out sets:

1. **Temporal test** — post-cutoff windows of training symbols (model saw the same stocks, different time)
2. **Held-out symbols** — 123 symbols never seen during training (tests cross-symbol generalisation)

Prerequisites: run `prepare_data.ipynb` (phases 1–5) to produce:
- `data/symbols.yml` — screened universe
- `data/splits.yml` — train/held-out assignment + per-symbol cutoffs
- `data/sentiment/<SYMBOL>.parquet` — pre-encoded sentiment
- `data/historical-prices/<SYMBOL>/<YEAR>.csv` — daily OHLCV
- `data/fundamentals/fundamentals.csv` — yfinance snapshots

In [1]:
from __future__ import annotations

import logging

import numpy as np
import pandas as pd
import torch
import yaml
from pathlib import Path
from torch.utils.data import DataLoader

import sentiment  # triggers load_dotenv()
from sentiment.log import setup_logging
from sentiment.sources.cache import MarketDataCache
from sentiment.sources.fundamental import FundamentalCache
from sentiment.embeddings import SentimentCache
from sentiment.features.technical import TechnicalFactors
from sentiment.features.dataloader import (
    FusedDataset, FusedStockDataset, build_dataset, make_loaders_multi,
)
from sentiment.model.lstm import SentimentLSTM
from sentiment.model.transformer import SentimentTransformer
from sentiment.model.train import train_model, bootstrap_evaluate

setup_logging()
logger = logging.getLogger("train_universal")

/Users/plouc314/Documents/finance/quant-sentiment-score/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

In [2]:
WINDOW      = 64
BATCH_SIZE  = 32
N_EPOCHS    = 100
MODEL_TYPE  = "lstm"  # "lstm" or "transformer"
PRICE_YEARS = [2018, 2019, 2020]
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

ROOT     = Path("..").resolve()
DATA_DIR = ROOT / "data"

print(f"Device: {DEVICE}")

Device: cpu


## Load Splits

Read the universe assignment and per-symbol cutoff dates produced by `prepare_data.ipynb` phase 5.

In [3]:
with open(DATA_DIR / "splits.yml") as f:
    splits = yaml.safe_load(f)

train_symbols    = splits["train_symbols"]
held_out_symbols = splits["held_out_symbols"]
cutoffs = {s: pd.Timestamp(d) for s, d in splits["cutoffs"].items()}
val_months = splits["config"]["val_months"]

print(f"Train symbols   : {len(train_symbols)}")
print(f"Held-out symbols: {len(held_out_symbols)}")
print(f"Split config    : {splits['config']}")

Train symbols   : 491
Held-out symbols: 123
Split config    : {'held_out_frac': 0.2, 'nominal_cutoff': '2019-10-01', 'seed': 42, 'stagger_days': 45, 'val_months': 2}


## Load Price Data

Concatenate yearly CSVs for each symbol across all `PRICE_YEARS`.
Symbols missing from cache for all years are recorded and excluded.

In [4]:
cache = MarketDataCache()


def _load_price(symbol: str, years: list[int]) -> pd.DataFrame | None:
    """Load and concatenate yearly price CSVs; return None if no data is found."""
    dfs = []
    for year in years:
        try:
            dfs.append(cache.load(symbol, year))
        except FileNotFoundError:
            pass
    if not dfs:
        return None
    df = pd.concat(dfs).sort_index()
    return df[~df.index.duplicated(keep="first")]


all_symbols = train_symbols + held_out_symbols
price_dfs: dict[str, pd.DataFrame | None] = {}
for symbol in all_symbols:
    price_dfs[symbol] = _load_price(symbol, PRICE_YEARS)

n_ok = sum(1 for v in price_dfs.values() if v is not None)
print(f"Price data loaded: {n_ok} / {len(all_symbols)} symbols")

Price data loaded: 614 / 614 symbols


## Load Fundamentals

In [5]:
fund_cache = FundamentalCache()

fund_dfs: dict[str, pd.DataFrame | None] = {}
for symbol in all_symbols:
    fund_dfs[symbol] = fund_cache.load_df(symbol)

n_ok = sum(1 for v in fund_dfs.values() if v is not None)
print(f"Fundamentals loaded: {n_ok} / {len(all_symbols)} symbols")

Fundamentals loaded: 614 / 614 symbols


## Load Sentiment

Reads per-symbol parquet files from `data/sentiment/`. Symbols without a cached
parquet will use zero vectors for sentiment inputs.

In [6]:
sentiment_cache = SentimentCache()

sentiment_dfs: dict[str, pd.DataFrame | None] = {}
for symbol in all_symbols:
    if sentiment_cache.exists(symbol):
        sentiment_dfs[symbol] = sentiment_cache.load(symbol)
    else:
        sentiment_dfs[symbol] = None

n_ok = sum(1 for v in sentiment_dfs.values() if v is not None)
print(f"Sentiment loaded: {n_ok} / {len(all_symbols)} symbols")

Sentiment loaded: 614 / 614 symbols


## Build Training Datasets

One `FusedDataset` per training symbol. Symbols with insufficient price history
(< 60 SMA warmup + window + 2 target rows) are skipped with a warning.

In [7]:
technical = TechnicalFactors()

train_datasets: list[tuple[str, FusedDataset]] = []
skipped_train: list[str] = []

for symbol in train_symbols:
    df = price_dfs[symbol]
    if df is None:
        skipped_train.append(symbol)
        continue
    try:
        ds = build_dataset(
            df,
            technical,
            sentiment_df=sentiment_dfs[symbol],
            ticker=symbol,
            window=WINDOW,
            fundamental_df=fund_dfs[symbol],
            include_momentum_slope=True,
        )
        train_datasets.append((symbol, ds))
    except RuntimeError as exc:
        logger.warning("Skipping %s: %s", symbol, exc)
        skipped_train.append(symbol)

print(f"Training datasets built: {len(train_datasets)} / {len(train_symbols)}")
if skipped_train:
    print(f"Skipped ({len(skipped_train)}): {skipped_train}")

16:30:05 INFO     sentiment.features.dataloader  Built dataset: 629 windows, 16 tech factors, 10 fundamental factors (incl. momentum slope), 3 sentiment prob features, window=64
16:30:05 INFO     sentiment.features.dataloader  Built dataset: 629 windows, 16 tech factors, 10 fundamental factors (incl. momentum slope), 3 sentiment prob features, window=64
16:30:05 INFO     sentiment.features.dataloader  Built dataset: 629 windows, 16 tech factors, 10 fundamental factors (incl. momentum slope), 3 sentiment prob features, window=64
16:30:05 INFO     sentiment.features.dataloader  Built dataset: 629 windows, 16 tech factors, 10 fundamental factors (incl. momentum slope), 3 sentiment prob features, window=64
16:30:05 INFO     sentiment.features.dataloader  Built dataset: 629 windows, 16 tech factors, 10 fundamental factors (incl. momentum slope), 3 sentiment prob features, window=64
16:30:05 INFO     sentiment.features.dataloader  Built dataset: 629 windows, 16 tech factors, 10 fundamental f

Training datasets built: 491 / 491


## DataLoaders

`make_loaders_multi` splits each symbol's windows at its own cutoff date:

- **train** — before `cutoff - val_months`
- **val**   — `[cutoff - val_months, cutoff)`
- **test**  — `>= cutoff`

Scalers are fitted on the concatenated training windows of all symbols.

In [8]:
train_loader, val_loader, test_loader, tech_scaler, fund_scaler = make_loaders_multi(
    train_datasets,
    cutoffs=cutoffs,
    val_months=val_months,
    batch_size=BATCH_SIZE,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")
print(f"Test  batches: {len(test_loader)}")

AttributeError: 'numpy.ndarray' object has no attribute 'values'

## Model

In [ ]:
# Derive architecture sizes from the first training dataset
_sample_ds = train_datasets[0][1]
n_fundamentals    = _sample_ds["X_fundamental"].shape[1]
n_sentiment_probs = _sample_ds["X_sentiment_probs"].shape[2]

if MODEL_TYPE == "lstm":
    model = SentimentLSTM(
        n_factors=16,
        sentiment_dim=768,
        hidden_size=32,
        num_layers=2,
        dropout=0.2,
        n_fundamentals=n_fundamentals,
        n_sentiment_probs=n_sentiment_probs,
    ).to(DEVICE)
else:
    model = SentimentTransformer(
        n_factors=16,
        sentiment_dim=768,
        d_model=64,
        nhead=4,
        n_layers=6,
        dim_feedforward=128,
        dropout=0.2,
        n_fundamentals=n_fundamentals,
        n_sentiment_probs=n_sentiment_probs,
    ).to(DEVICE)

print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

## Training

In [ ]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=N_EPOCHS,
    lr=1e-3,
    patience=15,
    device=DEVICE,
)

print(f"Best epoch: {history['best_epoch']}  |  Best val AUC: {history['best_val_auc']:.4f}")

## Evaluation — Temporal Test Set

Post-cutoff windows of training symbols: the model saw these stocks during training
but on different (earlier) time windows.

In [ ]:
result_temporal = bootstrap_evaluate(model, test_loader, device=DEVICE, n_bootstrap=1000, seed=42)

print("=== Temporal test set (training symbols, held-out time window) ===")
print(f"AUC      : {result_temporal['auc_mean']:.3f}  [{result_temporal['auc_ci_low']:.3f}, {result_temporal['auc_ci_high']:.3f}]")
print(f"Accuracy : {result_temporal['accuracy_mean']:.3f}  [{result_temporal['accuracy_ci_low']:.3f}, {result_temporal['accuracy_ci_high']:.3f}]")
print(f"Precision: {result_temporal['precision_mean']:.3f}  [{result_temporal['precision_ci_low']:.3f}, {result_temporal['precision_ci_high']:.3f}]")
print(f"Recall   : {result_temporal['recall_mean']:.3f}  [{result_temporal['recall_ci_low']:.3f}, {result_temporal['recall_ci_high']:.3f}]")

## Evaluation — Held-Out Symbols

Build datasets for the 123 symbols never seen during training, apply the scalers
fitted on training data, then evaluate the trained model.

In [ ]:
held_out_datasets: list[FusedDataset] = []
skipped_ho: list[str] = []

for symbol in held_out_symbols:
    df = price_dfs[symbol]
    if df is None:
        skipped_ho.append(symbol)
        continue
    try:
        ds = build_dataset(
            df,
            technical,
            sentiment_df=sentiment_dfs[symbol],
            ticker=symbol,
            window=WINDOW,
            fundamental_df=fund_dfs[symbol],
            include_momentum_slope=True,
        )
        held_out_datasets.append(ds)
    except RuntimeError as exc:
        logger.warning("Skipping held-out %s: %s", symbol, exc)
        skipped_ho.append(symbol)

print(f"Held-out datasets built: {len(held_out_datasets)} / {len(held_out_symbols)}")
if skipped_ho:
    print(f"Skipped: {skipped_ho}")

In [ ]:
# Concatenate all held-out windows and apply training scalers
ho_pool: FusedDataset = {
    "X_tech":            np.concatenate([d["X_tech"]            for d in held_out_datasets]),
    "X_sentiment":       np.concatenate([d["X_sentiment"]       for d in held_out_datasets]),
    "X_fundamental":     np.concatenate([d["X_fundamental"]     for d in held_out_datasets]),
    "X_sentiment_probs": np.concatenate([d["X_sentiment_probs"] for d in held_out_datasets]),
    "y":                 np.concatenate([d["y"]                 for d in held_out_datasets]),
    "dates":             np.concatenate([d["dates"]             for d in held_out_datasets]),
}

X_tech_ho = ho_pool["X_tech"].copy()
n, w, f = X_tech_ho.shape
X_tech_ho = (
    tech_scaler.transform(X_tech_ho.reshape(-1, f))
    .reshape(n, w, f)
    .astype(np.float32)
)

X_fund_ho = ho_pool["X_fundamental"].copy()
if fund_scaler is not None:
    X_fund_ho = fund_scaler.transform(X_fund_ho).astype(np.float32)

held_out_torch = FusedStockDataset(
    X_tech_ho,
    ho_pool["X_sentiment"],
    X_fund_ho,
    ho_pool["X_sentiment_probs"],
    ho_pool["y"],
)
held_out_loader = DataLoader(held_out_torch, batch_size=BATCH_SIZE, shuffle=False)

print(f"Held-out windows: {len(held_out_torch)}")

In [ ]:
result_held_out = bootstrap_evaluate(model, held_out_loader, device=DEVICE, n_bootstrap=1000, seed=42)

print("=== Held-out symbols (never seen during training) ===")
print(f"AUC      : {result_held_out['auc_mean']:.3f}  [{result_held_out['auc_ci_low']:.3f}, {result_held_out['auc_ci_high']:.3f}]")
print(f"Accuracy : {result_held_out['accuracy_mean']:.3f}  [{result_held_out['accuracy_ci_low']:.3f}, {result_held_out['accuracy_ci_high']:.3f}]")
print(f"Precision: {result_held_out['precision_mean']:.3f}  [{result_held_out['precision_ci_low']:.3f}, {result_held_out['precision_ci_high']:.3f}]")
print(f"Recall   : {result_held_out['recall_mean']:.3f}  [{result_held_out['recall_ci_low']:.3f}, {result_held_out['recall_ci_high']:.3f}]")

## Save Model

In [ ]:
save_path = ROOT / f"universal_{MODEL_TYPE}.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "tech_scaler": tech_scaler,
    "fund_scaler": fund_scaler,
    "config": {
        "model_type": MODEL_TYPE,
        "window": WINDOW,
        "n_fundamentals": n_fundamentals,
        "n_sentiment_probs": n_sentiment_probs,
    },
}, save_path)
print(f"Saved to {save_path}")